In [28]:
import sys
from pathlib import Path

EVAL_PIPELINE_DIR = Path("/home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/eval/Pipeline")
PROJECT_SRC_DIR = Path("/home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/src/rag")
sys.path.insert(0, str(EVAL_PIPELINE_DIR))
sys.path.insert(0, str(PROJECT_SRC_DIR))

from eval.dataset_loader import load_dataset
from eval.evaluators import EVALUATOR_REGISTRY
from eval.pipeline import run_pipeline, print_summary

from retriever import Retriever
from generator import OllamaGenerator

In [18]:
INDEX_PATH = "/home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/tests/virtual_database/index.faiss"
CONTEXTS_PATH = "/home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/tests/virtual_database/contexts.json"    # sửa đường dẫn thật
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GEN_MODEL = "qwen2.5:3b"

retriever = Retriever(index_path=INDEX_PATH, contexts_path=CONTEXTS_PATH, model_name=EMBED_MODEL)
generator = OllamaGenerator(model_name=GEN_MODEL)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14903.69it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
def predict_fn_retriever(input_data):
    query = input_data["query"]
    top_k = input_data.get("top_k", 5)
    return retriever.retrieve(query, top_k=top_k)
def predict_fn_rag(input_data):
    query = input_data["query"]
    retrieval_res = retriever.retrieve(query, top_k=5)

    if retrieval_res.get("metadata", {}).get("status") != "SUCCESS":
        return {"answer": "", "sources": [], "metadata": {"status": retrieval_res.get("metadata", {}).get("status")}}

    gen_result = generator.generate(query, retrieval_res["documents"])

    if gen_result["status"] != "SUCCESS":
        return {"answer": "", "sources": [], "metadata": {"status": gen_result["status"], "error": gen_result.get("error")}}

    sources = [
        {
            "document_id": c.get("chunk_id") or c.get("source", "unknown"),
            "content": c.get("content", ""),
            "metadata": {"source": c.get("source", "")},
        }
        for c in gen_result.get("citations", [])
    ]

    return {"answer": gen_result["answer"], "sources": sources, "metadata": {"status": "SUCCESS"}}

In [29]:
dataset = load_dataset(str(EVAL_PIPELINE_DIR / "data" / "retriever_sample.json"))
evaluator = EVALUATOR_REGISTRY["retriever"]()

report_retriever = run_pipeline(
    dataset=dataset,
    predict_fn=predict_fn_retriever,
    evaluator=evaluator,
    module_name="retriever",
    save_path=str(EVAL_PIPELINE_DIR / "reports" / "retriever_project_report.json"),
)
print_summary(report_retriever)

=== EVALUATION REPORT: retriever ===
Tổng số case : 2
Thành công   : 2
Lỗi          : 0
--- Điểm trung bình (toàn bộ) ---
  mrr                 : 0.0000
  recall@1            : 0.0000
  precision@1         : 0.0000
  ndcg@1              : 0.0000
  recall@3            : 0.0000
  precision@3         : 0.0000
  ndcg@3              : 0.0000
  recall@5            : 0.0000
  precision@5         : 0.0000
  ndcg@5              : 0.0000
  recall@10           : 0.0000
  precision@10        : 0.0000
  ndcg@10             : 0.0000
--- Breakdown theo case_type ---
  [normal]
    mrr               : 0.0000
    recall@1          : 0.0000
    precision@1       : 0.0000
    ndcg@1            : 0.0000
    recall@3          : 0.0000
    precision@3       : 0.0000
    ndcg@3            : 0.0000
    recall@5          : 0.0000
    precision@5       : 0.0000
    ndcg@5            : 0.0000
    recall@10         : 0.0000
    precision@10      : 0.0000
    ndcg@10           : 0.0000


In [31]:
dataset = load_dataset(str(EVAL_PIPELINE_DIR / "data" / "rag_sample.json"))
evaluator = EVALUATOR_REGISTRY["rag"]()

report_rag = run_pipeline(
    dataset=dataset,
    predict_fn=predict_fn_rag,
    evaluator=evaluator,
    module_name="rag",
    save_path=str(EVAL_PIPELINE_DIR / "reports" / "rag_project_report.json"),
)
print_summary(report_rag)

=== EVALUATION REPORT: rag ===
Tổng số case : 2
Thành công   : 2
Lỗi          : 0
--- Điểm trung bình (toàn bộ) ---
  exact_match         : 0.0000
  token_f1            : 0.0000
  rouge_l             : 0.0000
  bleu1               : 0.0000
  bleu2               : 0.0000
  bleu3               : 0.0000
  bleu4               : 0.0000
  source_recall       : 0.0000
--- Breakdown theo case_type ---
  [normal]
    exact_match       : 0.0000
    token_f1          : 0.0000
    rouge_l           : 0.0000
    bleu1             : 0.0000
    bleu2             : 0.0000
    bleu3             : 0.0000
    bleu4             : 0.0000
    source_recall     : 0.0000
  [edge]
    exact_match       : 0.0000
    token_f1          : 0.0000
    rouge_l           : 0.0000
    bleu1             : 0.0000
    bleu2             : 0.0000
    bleu3             : 0.0000
    bleu4             : 0.0000
    source_recall     : 0.0000
